# E7 — Two-Phase Curriculum Reward (MATH Competition)

**Tinker RL Project — PES University MTech Capstone (Group 6)**

| Field | Value |
|-------|-------|
| **Hypothesis** | Rewarding \\boxed{} format first (phase 1), then correctness (phase 2), breaks through the 14% MATH ceiling |
| **Model** | `Qwen/Qwen3-8B` |
| **Dataset** | Hendrycks MATH (all 7 subjects) |
| **Phase 1** | Steps 0–14: format reward (0/0.5/1.0 based on \\boxed{} presence/validity) |
| **Phase 2** | Steps 15–50: correctness reward (binary) |
| **Key metrics** | `train/format_score`, `train/correctness_score`, `train/current_phase` |
| **Baseline** | `math_qwen_8b.ipynb` result: ~14% at step 50 |
| **Finding it tests** | Finding 2: two-stage format→reasoning learning trajectory |

Override phase boundary: set `CURRICULUM_PHASE1_STEPS=<n>` env var (default 15).

In [ ]:
!pip install -q atroposlib==0.3.0 \
    'git+https://github.com/thinking-machines-lab/tinker.git' \
    datasets transformers wandb pydantic python-dotenv math-verify latex2sympy2-extended

In [ ]:
!git clone https://github.com/pes-llm-research/tinker-rl-lab.git
%cd tinker-rl-lab/atropos
!pip install -e . -q

In [ ]:
import os

os.environ['TINKER_API_KEY'] = 'YOUR_TINKER_API_KEY'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
os.environ['TINKER_CONFIG_PATH'] = 'configs/math_curriculum_qwen8b.yaml'

# Phase 1 lasts this many wandb_log cycles (~= training steps / steps_per_eval)
os.environ['CURRICULUM_PHASE1_STEPS'] = '15'

print('Config set. Phase 1 ends at step 15, Phase 2 runs steps 15-50.')

In [ ]:
!nohup python -m atroposlib.server > /tmp/atropos_server.log 2>&1 &
import time; time.sleep(5)
print('Atropos coordinator started')

In [ ]:
!nohup python -m tinker_atropos.environments.math_curriculum_tinker \
    --config configs/math_curriculum_qwen8b.yaml \
    > /tmp/math_curriculum_env.log 2>&1 &
import time; time.sleep(10)
print('Environment started')
!tail -5 /tmp/math_curriculum_env.log

In [ ]:
!python launch_training.py --config configs/math_curriculum_qwen8b.yaml

In [ ]:
# RESULTS — paste wandb data after training completes
# Compare against baseline (standard MATH reward, ~14% at step 50)

PHASE1_BOUNDARY = 15
baseline_final = 0.14  # from math_qwen_8b run

steps = []            # TODO
rewards = []          # TODO  (train/correctness_score — always tracks correctness)
format_scores = []    # TODO  (train/format_score)

if steps and rewards:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy with phase boundary
    ax1.plot(steps, rewards, color='#9b59b6', linewidth=2, label='Curriculum (this run)')
    ax1.axhline(y=baseline_final, color='gray', linestyle='--', alpha=0.6, label=f'Baseline (standard reward): {baseline_final:.0%}')
    ax1.axvline(x=PHASE1_BOUNDARY, color='orange', linestyle=':', linewidth=1.5, label=f'Phase transition (step {PHASE1_BOUNDARY})')
    ax1.set_xlabel('Training Step'); ax1.set_ylabel('Correctness (pass@1)'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax1.set_title('MATH Accuracy: Curriculum vs Baseline')
    ax1.set_ylim(-0.02, 0.6)

    # Format score over time
    ax2.plot(steps, format_scores, color='#e67e22', linewidth=2, label='Format score')
    ax2.axvline(x=PHASE1_BOUNDARY, color='orange', linestyle=':', linewidth=1.5, label='Phase transition')
    ax2.set_xlabel('Training Step'); ax2.set_ylabel('Format Score (\\boxed{} compliance)'); ax2.legend(); ax2.grid(True, alpha=0.3)
    ax2.set_title('\\\\boxed{} Format Compliance Over Training')
    ax2.set_ylim(-0.02, 1.1)

    plt.suptitle('E7: Two-Phase Curriculum Reward — Qwen3-8B on MATH', fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f'Final accuracy: {rewards[-1]:.2%}')
    print(f'Baseline:       {baseline_final:.2%}')
    print(f'Improvement:    +{rewards[-1]-baseline_final:.2%}')
    print(f'Hypothesis:     {"CONFIRMED" if rewards[-1] > baseline_final + 0.03 else "NOT CONFIRMED"}')
else:
    print('Run training first.')